<a href="https://colab.research.google.com/github/lswoodentoys/research_paper/blob/main/OECD_Tourism_Analysis_Notebook_version_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OECD Tourism Dataset Analysis

## 1. Cài đặt thư viện

In [2]:
%pip install pandas numpy matplotlib scipy statsmodels openpyxl


from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## 2. Nạp thư viện và cấu hình đường dẫn

Cập nhật đường dẫn CSV và thư mục xuất kết quả theo máy của bạn.

In [ ]:
INPUT_CSV = "/content/drive/MyDrive/Python_study/datasets/OECD_Tour_dataset.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Python_study/datasets/OECD_Tourism_Analysis_Results"


## 3. Hàm xử lý dữ liệu và phân tích

Cell này chứa các hàm đọc, làm sạch, pivot dữ liệu OECD, thống kê, tương quan, mô hình và vẽ biểu đồ.

In [8]:
"""
OECD Tourism Panel Analysis (2021–2025)
--------------------------------------
Input: OECD Tourism CSV in OECD SDMX long format.
Expected fields: REF_AREA, Reference area, MEASURE, Measure,
                 TIME_PERIOD, OBS_VALUE, Observation value.

Run locally:
    pip install pandas numpy matplotlib scipy statsmodels openpyxl
    python OECD_Tourism_Analysis.py --input "OECD_Tour_dataset(1).csv"

Run in Google Colab:
    1) Mount Drive and set INPUT_CSV to your Drive path.
    2) Run: %run /content/drive/MyDrive/.../OECD_Tourism_Analysis.py
       Or execute this file after uploading it to Colab.

Outputs: CSV tables, Excel workbook, PNG charts, audit report.
Interpretation: observational associations, not causal effects.
"""

from pathlib import Path
import argparse
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
import statsmodels.formula.api as smf

INDICATORS = ["GDP_SH", "GVA_SH", "EMP_SH"]
START_YEAR, END_YEAR = 2021, 2025

# Optional: set an explicit input path here for Colab, e.g.
# INPUT_CSV = "/content/drive/MyDrive/Python_study/datasets/OECD_Tour_dataset.csv"
INPUT_CSV = "/content/drive/MyDrive/Python_study/datasets/OECD_Tour_dataset.csv"


def find_column(df, candidates, required=True):
    """Return first matching column name, case-insensitive and whitespace-trimmed."""
    lookup = {str(c).strip().casefold(): c for c in df.columns}
    for candidate in candidates:
        key = candidate.strip().casefold()
        if key in lookup:
            return lookup[key]
    if required:
        raise ValueError(
            f"Could not find any of {candidates}. Available columns: {df.columns.tolist()}"
        )
    return None


def read_oecd_csv(path):
    """Read OECD CSV and identify its long-format columns."""
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(
            f"Input CSV not found: {path}\n"
            "Check the path. In Colab, mount Google Drive and set INPUT_CSV."
        )

    # Standard OECD downloads are usually comma-delimited. Fall back to delimiter sniffing.
    try:
        raw = pd.read_csv(path, low_memory=False)
        if raw.shape[1] == 1:
            raw = pd.read_csv(path, sep=None, engine="python", low_memory=False)
    except Exception:
        raw = pd.read_csv(path, sep=None, engine="python", low_memory=False)

    print(f"Loaded: {path}")
    print(f"Raw shape: {raw.shape}")
    print("Columns:", raw.columns.tolist())

    country_code_col = find_column(raw, ["REF_AREA"], required=False)
    country_name_col = find_column(raw, ["Reference area"], required=False)
    measure_col = find_column(raw, ["MEASURE"], required=False)
    measure_name_col = find_column(raw, ["Measure"], required=False)
    year_col = find_column(raw, ["TIME_PERIOD", "Time period"])
    value_col = find_column(raw, ["OBS_VALUE", "Observation value"])

    if country_code_col is None and country_name_col is None:
        raise ValueError("No country field found (expected REF_AREA or Reference area).")
    if measure_col is None and measure_name_col is None:
        raise ValueError("No measure field found (expected MEASURE or Measure).")

    # Prefer stable OECD codes over descriptive labels.
    country_col = country_code_col or country_name_col
    indicator_col = measure_col or measure_name_col

    use_cols = [country_col, year_col, indicator_col, value_col]
    clean = raw[use_cols].copy()
    clean.columns = ["country", "year", "indicator", "value"]

    clean["country"] = clean["country"].astype("string").str.strip()
    clean["indicator"] = clean["indicator"].astype("string").str.strip().str.upper()
    clean["year"] = pd.to_numeric(clean["year"], errors="coerce")
    clean["value"] = pd.to_numeric(clean["value"], errors="coerce")

    # Remove rows with unusable country/year and keep requested period/indicators.
    clean = clean.dropna(subset=["country", "year"])
    clean["year"] = clean["year"].astype(int)
    clean = clean[
        clean["year"].between(START_YEAR, END_YEAR)
        & clean["indicator"].isin(INDICATORS)
    ].copy()

    if clean.empty:
        raise ValueError(
            "No records for GDP_SH, GVA_SH, EMP_SH in 2021–2025. "
            "Check the file, measure codes, and year range."
        )

    # Detect duplicate country-year-indicator keys before pivoting.
    key_cols = ["country", "year", "indicator"]
    duplicate_rows = clean[clean.duplicated(key_cols, keep=False)].sort_values(key_cols)

    # If duplicates exist, identical values are collapsed; conflicting values stop analysis.
    if not duplicate_rows.empty:
        conflicts = (
            duplicate_rows.groupby(key_cols, dropna=False)["value"]
            .nunique(dropna=False)
            .reset_index(name="distinct_values")
        )
        conflicts = conflicts[conflicts["distinct_values"] > 1]
        if not conflicts.empty:
            raise ValueError(
                "Conflicting duplicate country-year-indicator records were found. "
                "Review duplicate keys in the source CSV before analysis.\n"
                + conflicts.head(20).to_string(index=False)
            )
        clean = clean.drop_duplicates(key_cols, keep="first")
        warnings.warn("Identical duplicate records were collapsed.")

    # Wide panel: one row per country-year and one column per indicator.
    panel = clean.pivot(index=["country", "year"], columns="indicator", values="value")
    panel = panel.reindex(columns=INDICATORS).reset_index()
    panel.columns.name = None

    # Country labels, when available, are retained as a separate lookup.
    if country_code_col is not None and country_name_col is not None:
        labels = raw[[country_code_col, country_name_col]].dropna().drop_duplicates()
        labels.columns = ["country", "country_name"]
        panel = panel.merge(labels, on="country", how="left", validate="many_to_one")

    return raw, clean, panel, duplicate_rows


def make_descriptives(panel):
    indicators = panel[INDICATORS]
    desc = indicators.describe(percentiles=[0.25, 0.50, 0.75]).T
    desc = desc.rename(columns={"25%": "Q1", "50%": "Median", "75%": "Q3"})
    annual = panel.groupby("year")[INDICATORS].agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
    annual.columns = [f"{var}_{stat}" for var, stat in annual.columns]
    return desc, annual.reset_index()


def correlation_table(panel):
    rows = []
    for xvar in ["GDP_SH", "GVA_SH"]:
        pair = panel[[xvar, "EMP_SH"]].dropna()
        for method, fn in [("Pearson", pearsonr), ("Spearman", spearmanr)]:
            if len(pair) < 3 or pair[xvar].nunique() < 2 or pair["EMP_SH"].nunique() < 2:
                r, p = np.nan, np.nan
            else:
                r, p = fn(pair[xvar], pair["EMP_SH"])
            rows.append({
                "predictor": xvar, "outcome": "EMP_SH", "method": method,
                "n": len(pair), "correlation": r, "p_value": p
            })

        # Within-country correlation after subtracting each country's mean.
        within = panel[["country", xvar, "EMP_SH"]].dropna().copy()
        within["x_dm"] = within[xvar] - within.groupby("country")[xvar].transform("mean")
        within["y_dm"] = within["EMP_SH"] - within.groupby("country")["EMP_SH"].transform("mean")
        for method, fn in [("Pearson", pearsonr), ("Spearman", spearmanr)]:
            if len(within) < 3 or within["x_dm"].nunique() < 2 or within["y_dm"].nunique() < 2:
                r, p = np.nan, np.nan
            else:
                r, p = fn(within["x_dm"], within["y_dm"])
            rows.append({
                "predictor": xvar, "outcome": "EMP_SH", "method": method,
                "scope": "Within-country demeaned", "n": len(within),
                "correlation": r, "p_value": p
            })
    result = pd.DataFrame(rows)
    if "scope" not in result.columns:
        result["scope"] = "Pooled"
    result["scope"] = result["scope"].fillna("Pooled")
    return result[["scope", "predictor", "outcome", "method", "n", "correlation", "p_value"]]


def fit_model(panel, predictor, model_label, country_fe=False, year_fe=False):
    d = panel[["country", "year", predictor, "EMP_SH"]].dropna().copy()
    if d.empty:
        return None, d

    formula = f"EMP_SH ~ {predictor}"
    if country_fe:
        formula += " + C(country)"
    if year_fe:
        formula += " + C(year)"

    # Cluster-robust SEs by country. Use only if multiple country clusters exist.
    n_clusters = d["country"].nunique()
    if n_clusters >= 2:
        result = smf.ols(formula, data=d).fit(
            cov_type="cluster", cov_kwds={"groups": d["country"]}
        )
    else:
        warnings.warn(f"Only {n_clusters} country cluster(s); using conventional SEs.")
        result = smf.ols(formula, data=d).fit()
    return result, d


def regression_tables(panel):
    rows = []
    model_objects = {}
    specs = [
        ("Pooled OLS", False, False),
        ("Country FE", True, False),
        ("Two-way FE", True, True),
    ]
    for predictor in ["GDP_SH", "GVA_SH"]:
        for label, country_fe, year_fe in specs:
            result, d = fit_model(panel, predictor, label, country_fe, year_fe)
            if result is None or predictor not in result.params.index:
                continue
            model_objects[f"{predictor}_{label}"] = result
            lo, hi = result.conf_int().loc[predictor].tolist()
            rows.append({
                "predictor": predictor, "model": label,
                "coefficient": result.params[predictor],
                "std_error": result.bse[predictor],
                "p_value": result.pvalues[predictor],
                "ci95_lower": lo, "ci95_upper": hi,
                "n_observations": int(result.nobs),
                "n_countries": d["country"].nunique(),
                "r_squared": result.rsquared,
                "adjusted_r_squared": result.rsquared_adj,
            })
    return pd.DataFrame(rows), model_objects


def sensitivity_table(panel):
    rows = []
    for predictor in ["GDP_SH", "GVA_SH"]:
        result, d = fit_model(panel[panel["year"] != 2021], predictor,
                              "Exclude 2021", True, True)
        if result is not None and predictor in result.params.index:
            lo, hi = result.conf_int().loc[predictor].tolist()
            rows.append({
                "check": "Exclude 2021", "predictor": predictor,
                "coefficient": result.params[predictor], "std_error": result.bse[predictor],
                "p_value": result.pvalues[predictor], "ci95_lower": lo, "ci95_upper": hi,
                "n_observations": int(result.nobs), "n_countries": d["country"].nunique()
            })

    complete = panel.dropna(subset=INDICATORS).copy()
    expected = set(range(START_YEAR, END_YEAR + 1))
    year_sets = complete.groupby("country")["year"].apply(lambda s: set(s.astype(int)))
    balanced_ids = year_sets[year_sets.apply(lambda years: expected.issubset(years))].index
    balanced = complete[complete["country"].isin(balanced_ids)].copy()
    print(f"Countries in complete balanced panel: {len(balanced_ids)}")

    for predictor in ["GDP_SH", "GVA_SH"]:
        result, d = fit_model(balanced, predictor, "Balanced panel", True, True)
        if result is not None and predictor in result.params.index:
            lo, hi = result.conf_int().loc[predictor].tolist()
            rows.append({
                "check": "Balanced panel", "predictor": predictor,
                "coefficient": result.params[predictor], "std_error": result.bse[predictor],
                "p_value": result.pvalues[predictor], "ci95_lower": lo, "ci95_upper": hi,
                "n_observations": int(result.nobs), "n_countries": d["country"].nunique()
            })
    return pd.DataFrame(rows)


def create_charts(panel, output_dir):
    """Create annual trend, scatter, and country-level comparison charts."""
    output_dir.mkdir(parents=True, exist_ok=True)
    charts = []

    # Chart 1: annual average of each indicator.
    annual_mean = panel.groupby("year")[INDICATORS].mean(numeric_only=True)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for col in INDICATORS:
        ax.plot(annual_mean.index, annual_mean[col], marker="o", linewidth=2, label=col)
    ax.set_title("Average direct tourism contribution by year")
    ax.set_xlabel("Year")
    ax.set_ylabel("Indicator value (check source unit)")
    ax.set_xticks(sorted(panel["year"].unique()))
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    f = output_dir / "01_annual_mean_trends.png"
    fig.savefig(f, dpi=180)
    plt.close(fig)
    charts.append(f)

    # Chart 2 and 3: pooled scatter with fitted linear trend.
    for xvar, num in [("GDP_SH", "02"), ("GVA_SH", "03")]:
        d = panel[[xvar, "EMP_SH"]].dropna()
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
        ax.scatter(d[xvar], d["EMP_SH"], alpha=0.65)
        if len(d) >= 2 and d[xvar].nunique() >= 2:
            slope, intercept = np.polyfit(d[xvar], d["EMP_SH"], 1)
            xline = np.linspace(d[xvar].min(), d[xvar].max(), 100)
            ax.plot(xline, intercept + slope * xline, linewidth=2)
        ax.set_title(f"Tourism {xvar} vs tourism employment share")
        ax.set_xlabel(xvar)
        ax.set_ylabel("EMP_SH")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        f = output_dir / f"{num}_scatter_{xvar}_vs_EMP_SH.png"
        fig.savefig(f, dpi=180)
        plt.close(fig)
        charts.append(f)

    # Chart 4: country-level mean values, only countries with enough observations.
    country_means = panel.groupby("country")[INDICATORS].mean()
    country_counts = panel.groupby("country")["year"].nunique()
    country_means = country_means.loc[country_counts[country_counts >= 3].index]
    if not country_means.empty:
        country_means = country_means.sort_values("EMP_SH", ascending=False).head(20)
        fig, ax = plt.subplots(figsize=(10, max(5, 0.32 * len(country_means))))
        country_means["EMP_SH"].sort_values().plot(kind="barh", ax=ax)
        ax.set_title("Mean tourism employment share by country (top 20 by value)")
        ax.set_xlabel("EMP_SH")
        ax.set_ylabel("Country code")
        ax.grid(axis="x", alpha=0.3)
        fig.tight_layout()
        f = output_dir / "04_country_mean_EMP_SH_top20.png"
        fig.savefig(f, dpi=180)
        plt.close(fig)
        charts.append(f)

    return charts


def write_audit(raw, clean, panel, duplicate_rows, output_dir, input_path):
    report = [
        f"Input file: {input_path}",
        f"Raw rows: {len(raw)}",
        f"Analysis long-format rows (2021–2025, selected measures): {len(clean)}",
        f"Panel rows after pivot: {len(panel)}",
        f"Countries: {panel['country'].nunique()}",
        f"Years: {sorted(panel['year'].dropna().unique().tolist())}",
        f"Rows in duplicate-key groups: {len(duplicate_rows)}",
        "",
        "Missing values in panel indicators:",
        panel[INDICATORS].isna().sum().to_string(),
        "",
        "Indicator ranges:",
        panel[INDICATORS].agg(["min", "max"]).T.to_string(),
        "",
        "Check source metadata for exact indicator definitions and units.",
        "Regression coefficients are associations, not causal effects."
    ]
    (output_dir / "audit_report.txt").write_text("\n".join(report), encoding="utf-8")


def main(input_path, output_path):
    output_dir = Path(output_path).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)

    raw, clean, panel, duplicate_rows = read_oecd_csv(input_path)
    print("\nPanel preview:")
    print(panel.head(10).to_string(index=False))

    descriptive, annual = make_descriptives(panel)
    correlations = correlation_table(panel)
    regressions, model_objects = regression_tables(panel)
    sensitivity = sensitivity_table(panel)
    charts = create_charts(panel, output_dir)

    # Export CSV tables.
    tables = {
        "panel_wide": panel,
        "descriptive_statistics": descriptive,
        "annual_summary": annual,
        "correlations": correlations,
        "regression_results": regressions,
        "sensitivity_results": sensitivity,
        "duplicate_records": duplicate_rows,
    }
    for name, table in tables.items():
        table.to_csv(output_dir / f"{name}.csv", index=False)

    # Export Excel workbook.
    excel_path = output_dir / "OECD_Tourism_Analysis_Results.xlsx"
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for name, table in tables.items():
            table.to_excel(writer, sheet_name=name[:31], index=False)

    write_audit(raw, clean, panel, duplicate_rows, output_dir, input_path)

    print("\nAnalysis completed.")
    print("Output folder:", output_dir.resolve())
    print("Excel workbook:", excel_path.resolve())
    print("Charts:")
    for chart in charts:
        print(" -", chart.name)


## 4. Chạy toàn bộ phân tích

Cell này gọi quy trình chính và xuất bảng/biểu đồ vào `OUTPUT_DIR`.

In [9]:
INPUT_CSV = "/content/drive/MyDrive/Python_study/datasets/OECD_Tour_dataset.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Python_study/datasets/OECD_Tourism_Analysis_Results"

print("File dữ liệu:", INPUT_CSV)
print("Thư mục kết quả:", OUTPUT_DIR)

File dữ liệu: /content/drive/MyDrive/Python_study/datasets/OECD_Tour_dataset.csv
Thư mục kết quả: /content/drive/MyDrive/Python_study/datasets/OECD_Tourism_Analysis_Results


In [10]:
from pathlib import Path

print("OUTPUT_DIR =", repr(OUTPUT_DIR))

folder = Path(OUTPUT_DIR)
print("Thư mục tồn tại:", folder.exists())

if folder.exists():
    files = list(folder.rglob("*"))
    print("Số mục trong thư mục:", len(files))

    for item in files:
        print(item)
else:
    print("Không tìm thấy thư mục tại đường dẫn này.")

OUTPUT_DIR = '/content/drive/MyDrive/Python_study/datasets/OECD_Tourism_Analysis_Results'
Thư mục tồn tại: False
Không tìm thấy thư mục tại đường dẫn này.


In [11]:
from pathlib import Path

for base in [Path("/content"), Path("/content/drive/MyDrive")]:
    print(f"\nTìm file kết quả trong: {base}")

    if base.exists():
        matches = [
            p for p in base.rglob("*")
            if p.is_file()
            and (
                "OECD_Tourism" in p.name
                or p.suffix.lower() in [".png", ".xlsx", ".csv", ".txt"]
            )
        ]

        for p in matches[:100]:
            print(p)


Tìm file kết quả trong: /content
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_train_small.csv

Tìm file kết quả trong: /content/drive/MyDrive
